# modify fits header

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
#import sys
#!pip install photutils==1.6

In [4]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, ysfitsutilpy, ysphotutilpy, astroscrappy, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        #print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module ysfitsutilpy is installed
**** module ysphotutilpy is installed
**** module astroscrappy is installed
**** module version_information is installed
This notebook was generated at 2023-09-03 16:51:49 (KST = GMT+0900) 


/home/guitar79/anaconda3/envs/astro_Python_env/lib/python3.11/site-packages/ysphotutilpy/seputil.py:113: UserWarning: Package sep is not installed. Some functions will not work.
  warn("Package sep is not installed. Some functions will not work.")


0 Python     3.11.4 64bit [GCC 11.2.0]
1 IPython    8.12.2
2 OS         Linux 5.15.0 82 generic x86_64 with glibc2.31
3 numpy      1.25.2
4 pandas     2.0.3
5 matplotlib 3.7.1
6 scipy      1.11.1
7 astropy    5.2.1
8 photutils  1.6.0
9 ccdproc    2.4.1
10 ysfitsutilpy 0.2
11 ysphotutilpy 0.1.1
12 astroscrappy 1.1.0
13 version_information 1.0.4


### import modules

In [10]:
from glob import glob
from pathlib import Path, PosixPath, WindowsPath
import os
from datetime import datetime
from astropy.io import fits
import shutil 

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

import _Python_utilities
import _astro_utilities

In [7]:
#%%
#######################################################
# read all files in base directory for processing
BASEDIR = Path(r"r:\CCD_obs")
BASEDIR = Path("/mnt/Rdata/OBS_data") 
#BASEDIR = Path("/mnt/OBS_data") 
DOINGDIR = Path(BASEDIR/ _astro_utilities.CCD_NEW_dir)

#DOINGDIR = Path(BASEDIR/ _astro_utilities.CCD_obs_raw_dir)

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfallsubDirs(DOINGDIR))
#print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))
#######################################################
            

len(DOINGDIRs):  49


In [8]:
#%%
for DOINGDIR in DOINGDIRs[:1] :
    DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        print(f"Starting: {str(DOINGDIR.parts[-1])}")
        summary = None 
        summary = yfu.make_summary(DOINGDIR/"*.fit*",
                    #output = save_fpath,
                    verbose = False
                    )
        print("summary: ", summary)
        print("len(summary)", len(summary))

DOINGDIR /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_-_2023-06-14_-_GSON300_SBIG STF-8300 CCD Camera-_-_-
len(fits_in_dir) 12
Starting: A847PA_LIGHT_-_2023-06-14_-_GSON300_SBIG STF-8300 CCD Camera-_-_-
summary:                                                   file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
1   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
2   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
3   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
4   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
5   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
6   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
7   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
8   /mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_...  16980480    True   
9   /mnt/Rdata/OBS_data/CC

In [9]:
for _, row in summary.iterrows():
    # 파일명 출력
    print (row["file"])
    fpath = Path(row["file"])
    try:
        hdul = _astro_utilities.KevinFitsUpdater(fpath)
        print("hdul: ", hdul)

    except Exception as err :
        print("X"*60)
        #_Python_utilities.write_log(err_log_file, err)
        print(err)

/mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_-_2023-06-14_-_GSON300_SBIG STF-8300 CCD Camera-_-_-/A847PA_-LIGHT_r_2023-06-14_20-58-44_90.00_GSON300_SBIG STF-8300 CCD Camera_-10.20_1x1GSON300.fits
foldername_el ['A847PA', 'LIGHT', '-', '2023-06-14', '-', 'GSON300', 'SBIG STF-8300 CCD Camera-', '-', '-']
fname_el ['A847PA', '-LIGHT', 'r', '2023-06-14', '20-58-44', '90.00', 'GSON300', 'SBIG STF-8300 CCD Camera', '-10.20', '1x1GSON300.fits']
object_name A847PA
optic_name GSON300
ccd_name SBIG STF-8300 CCD Camera-
OBJECT:  A847PA
TELESCOP:  GSON300
OPTIC:  None
CCDNAME:  None
FILTER:  r
GAIN:  None
EGAIN:  0.0
RDNOISE:  None
FOCALLEN:  1200.0
FOCRATIO:  4.0
PIXSCALE:  None
CCD-TEMP:  -10.2030840367361
XBINNING:  1
YBINNING:  1
FLIPSTAT:  None
EXPTIME:  90.0
EXPOSURE:  90.0
The 'OBJECT' is set A847PA
CCDNAME STF-8300M
The 'CCDNAME' is set STF-8300M...
The 'IMAGETYP' is set LIGHT
The 'OPTIC' is set GSON300
The 'FOCALLEN' is set 1200...
The 'FOCRATIO' is set 4.0...
GSON300_STF-8300M
The 'PIX

In [17]:
print(str(DOINGDIR))
print(str(BASEDIR/_astro_utilities.CCD_NEWUP_dir / DOINGDIR.stem))
#DOINGDIR.stem

shutil.move(str(DOINGDIR), str(BASEDIR/_astro_utilities.CCD_NEWUP_dir / DOINGDIR.stem))
print(str(DOINGDIR), str(BASEDIR/_astro_utilities.CCD_NEWUP_dir / DOINGDIR.stem))

/mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_-_2023-06-14_-_GSON300_SBIG STF-8300 CCD Camera-_-_-
/mnt/Rdata/OBS_data/CCD_newUpdated_files/A847PA_LIGHT_-_2023-06-14_-_GSON300_SBIG STF-8300 CCD Camera-_-_-
/mnt/Rdata/OBS_data/CCD_new_files/A847PA_LIGHT_-_2023-06-14_-_GSON300_SBIG STF-8300 CCD Camera-_-_- /mnt/Rdata/OBS_data/CCD_newUpdated_files/A847PA_LIGHT_-_2023-06-14_-_GSON300_SBIG STF-8300 CCD Camera-_-_-


In [18]:

#shutil.move(str(new_fpath), str(duplicate_fpath))

Starting: Cal
summary None
Starting: -_Bias_-_2016-10-05_-_-_QSI683ws_-_1bin
summary                                                  file  filesize  SIMPLE  \
0   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
1   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
2   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
3   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
4   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
5   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
6   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
7   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
8   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
9   R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
10  R:\CCD_obs\CCD_obs_raw\QSI683ws_1bin\Cal\-_Bia...  16663680    True   
11  R:\CCD_obs\